# Preprocessing steps

## Loading and normalizing the data

In [10]:
from datasets import load_dataset 
ds = load_dataset("bigcode/bigcodebench") 

In [19]:
split_name = list(ds.keys())[-1]
print("Using split:", split_name)
dataset_split = ds[split_name] 
print(dataset_split[0].keys())

Using split: v0.1.4
dict_keys(['task_id', 'complete_prompt', 'instruct_prompt', 'canonical_solution', 'code_prompt', 'test', 'entry_point', 'doc_struct', 'libs'])


In [ ]:
import uuid
import json

def normalize(dataset_split):
    normalized = []
    for entry in dataset_split:
        # doc_struct may be a JSON string
        doc_struct_raw = entry.get("doc_struct", "{}")
        try:
            doc_struct = json.loads(doc_struct_raw)
        except json.JSONDecodeError:
            doc_struct = {}

        # Extract description list
        description_list = doc_struct.get("description", [])
        description_text = " ".join(description_list)  # join into a single string

        normalized.append({
            "id": str(uuid.uuid4()),
            "language": entry.get("language", "python"),
            "original_code": entry.get("canonical_solution", ""),
            "test": [entry.get("test", "")],
            "description": description_text,  # new column
            "metadata": {
                "task_id": entry.get("task_id"),
                "libs": entry.get("libs", [])
            },
            "clones": []
        })
    return normalized 

normalized_data = normalize(dataset_split)

# Quick check
import json
print(json.dumps(normalized_data[1000], indent=2))


{
  "id": "23f620f1-b195-4f46-a7de-b4bd6782ddcd",
  "language": "python",
  "original_code": "import urllib.request\nimport os\nimport json\nimport pandas as pd\n# Constants\nTARGET_JSON_FILE = \"downloaded_file.json\"\ndef task_func(url):\n",
  "unit_tests": [
    "import unittest\nimport pandas as pd\nfrom unittest.mock import patch, mock_open\nclass TestCases(unittest.TestCase):\n    \"\"\"Test cases for the task_func function.\"\"\"\n    @patch(\"urllib.request.urlretrieve\")\n    @patch(\"os.remove\")\n    def test_sample_1(self, mock_remove, mock_urlretrieve):\n        \"\"\"Test that the function returns the correct DataFrame for a given JSON file.\"\"\"\n        url = \"http://example.com/sample_1.json\"\n        sample_data = '[{\"name\": \"Alice\", \"age\": 25, \"city\": \"New York\"}, {\"name\": \"Bob\", \"age\": 30, \"city\": \"San Francisco\"}]'\n        mock_urlretrieve.return_value = None\n        with patch(\"builtins.open\", mock_open(read_data=sample_data)):\n        